In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
base = ROOT / "data/raw/202611"
load_dotenv(ROOT / ".env")

cli = pd.read_csv(f"{base}/train_clientes_sample.csv")
req = pd.read_csv(f"{base}/train_requerimientos_sample.csv")
cli_oot = pd.read_csv(f"{base}/oot_clientes_sample.csv")
req_oot = pd.read_csv(f"{base}/oot_requerimientos_sample.csv")

In [7]:
# 1. Granularidad y periodos
print("CODMES clientes train:", sorted(cli.CODMES.unique()))
print("CODMES req train:", sorted(req.CODMES.unique()))
print("CODMES clientes OOT:", sorted(cli_oot.CODMES.unique()))
print("CODMES req OOT:", sorted(req_oot.CODMES.unique()))
print("¿ID único en clientes?", cli.ID_CORRELATIVO.is_unique)

CODMES clientes train: [np.int64(201208)]
CODMES req train: [np.int64(201208), np.int64(202403), np.int64(202404), np.int64(202405), np.int64(202406), np.int64(202407)]
CODMES clientes OOT: [np.int64(202408)]
CODMES req OOT: [np.int64(202403), np.int64(202404), np.int64(202405), np.int64(202406), np.int64(202407), np.int64(202408)]
¿ID único en clientes? True


In [8]:
# 2. Target y nulos
print("Tasa de fuga:", cli.ATTRITION.mean().round(4))
print(cli.isna().sum()[cli.isna().sum() > 0])
print(req.isna().sum()[req.isna().sum() > 0])

Tasa de fuga: 0.1546
RANG_INGRESO           7550
FLAG_LIMA_PROVINCIA    2723
EDAD                   4253
ANTIGUEDAD             1401
dtype: int64
DICTAMEN    1
dtype: int64


In [9]:
# 3. Cardinalidad de categóricas de requerimientos
for c in ["TIPO_REQUERIMIENTO2", "DICTAMEN", "PRODUCTO_SERVICIO_2", "SUBMOTIVO_2"]:
    print(c, req[c].nunique(), req[c].value_counts().head(5).to_dict())

TIPO_REQUERIMIENTO2 2 {'Reclamo': 30021, 'Solicitud': 11292}
DICTAMEN 3 {'PROCEDE TOTAL': 20224, 'NO PROCEDE': 19047, 'PROCEDE PARCIAL': 2041}
PRODUCTO_SERVICIO_2 22 {'Producto 20': 18332, 'Producto 07': 7531, 'Producto 18': 5644, 'Producto 21': 3437, 'Producto 17': 2366}
SUBMOTIVO_2 81 {'Submotivo 125': 15192, 'Submotivo 157': 4212, 'Submotivo 144': 4035, 'Submotivo 132': 2735, 'Submotivo 88': 1728}


In [10]:
# 4. Cobertura del cruce
print("% clientes con requerimientos:", cli.ID_CORRELATIVO.isin(req.ID_CORRELATIVO).mean().round(3))

% clientes con requerimientos: 0.502


In [ ]:
man_path = ROOT / "docs/manifiesto_datos.json"
man = json.loads(man_path.read_text(encoding="utf-8"))

for f in man["files"]:
    p = base / f["filename"]
    f["sha256"] = hashlib.sha256(p.read_bytes()).hexdigest()
    if p.suffix == ".csv":
        f["row_count"] = len(pd.read_csv(p))
    f["source_uri"] = f"local://{p.as_posix()}"  # se reemplaza por s3:// en el paso de AWS
man["dataset_release"] = "202611"
man["recorded_at_utc"] = datetime.now(timezone.utc).isoformat()
man_path.write_text(json.dumps(man, indent=2, ensure_ascii=False), encoding="utf-8")

2740

In [11]:
print(set(cli.columns) - set(cli_oot.columns))  # debería ser solo {'ATTRITION'}
print(set(cli_oot.columns) - set(cli.columns))  # debería ser vacío
print(cli_oot.ID_CORRELATIVO.is_unique, cli_oot.shape)

{'ATTRITION'}
set()
True (14000, 52)


In [ ]:
mlflow.set_experiment("/Shared/utec-bank-attrition")
with mlflow.start_run(run_name="smoke-test"):
    mlflow.log_param("test", 1)
    mlflow.log_metric("ok", 1.0)
print("OK:", mlflow.get_tracking_uri())

c:\Users\Renzo\anaconda3\envs\mlops-utec\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/10 21:01:09 INFO mlflow.tracking.fluent: Experiment with name '/Shared/utec-bank-attrition' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog


🏃 View run smoke-test at: https://dbc-f14abea7-4a83.cloud.databricks.com/ml/experiments/850869932413791/runs/8dd119679b9f4e76baa8e3036143a3ff
🧪 View experiment at: https://dbc-f14abea7-4a83.cloud.databricks.com/ml/experiments/850869932413791
OK: databricks
